# 送電線増強計画 + 運用の同時決定 — Transmission Expansion Planning

系統計画者は数十年単位で固定される「送電投資」と、その後の毎時の「発電配分(運用)」を
同時に検討しなければならない。既存の送電網は容量が小さく慢性的に混雑しており、
どの候補コリドー(送電線)を増強するかで、複数の需要シナリオ(猛暑ピーク・平常時・
低需要期)それぞれの運用可能性・コストが変わる。増強しなければ、その候補線には
物理法則(潮流=感受率×位相角差)自体が働かない — 建設可否が物理法則の有効/無効を
切り替える **disjunctive(big-M)構造**が、この問題の核心的な難しさである。

この notebook は次の型でこの問題を追う。

1. **素朴な定式化** — 候補線ごとの big-M disjunctive 制約とシナリオ別DC潮流の規模を確認
2. **診断** — `mk.analyze` で観測し、どの症状が発火するかを見る
3. **診断の中身を見る** — 係数スケール・対称性の収集器を直接叩いて発火根拠を裏付ける
4. **改善を試す** — 数値スケールfindingsのrecipe(Big-M排除)を実際に試し、効果を正直に測る

題材: `samples/location_and_network_design/transmission_expansion_operation.py`

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" の有無で探すと docs で止まる → pyproject.toml が目印。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/location_and_network_design"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum, SCIP_PARAMSETTING

def show(fig):  # 静的サイトに埋め込める形で表示(CDN の plotly.js、requirejsシムを避ける)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import transmission_expansion_operation as tep
from minlpkit.collectors.static_diag import extract_coefficients, scale_summary, residual_scale
from minlpkit.collectors.symmetry import detect_symmetry

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 素朴な定式化

`build_model("default")` はバス9・候補コリドー8・需要シナリオ5の規模。候補線ごとに
4本の disjunctive 制約(`kirchhoff_cand_ub/lb`・`cand_cap_ub/lb`)がシナリオ数だけ複製され、
共通の `BIG_M=400` で「増強しなければ潮流=0、物理法則も無効化」を表現している。

In [2]:
m = tep.build_model("default")
nB, nE, nCand, nS = m.data["dims"]
n_bin = sum(1 for v in m.getVars() if v.vtype() == "BINARY")
n_cont = sum(1 for v in m.getVars() if v.vtype() != "BINARY")
print(f"バス数={nB} 既存線={nE} 候補線={nCand} シナリオ数={nS}")
print(f"変数: 整数(増強決定) {n_bin} 個 / 連続(潮流・位相角・発電・停電) {n_cont} 個")
print(f"制約: {m.getNConss()} 本(内 disjunctive big-M制約 = 候補線数×シナリオ数×4本 = "
      f"{nCand * nS * 4} 本)")
print(f"共通のBIG_M = {tep.BIG_M}(全候補線・全シナリオで同一の定数)")

バス数=9 既存線=8 候補線=8 シナリオ数=5
変数: 整数(増強決定) 8 個 / 連続(潮流・位相角・発電・停電) 185 個
制約: 245 本(内 disjunctive big-M制約 = 候補線数×シナリオ数×4本 = 160 本)
共通のBIG_M = 400.0(全候補線・全シナリオで同一の定数)


共通の `BIG_M=400` は「どの候補線・どのシナリオでも安全に潮流をゼロ化できる」ための
最も緩い値であり、実際に個々の候補線が取りうる潮流の絶対値(感受率×位相角差の上限)は
はるかに小さい。この「本当に必要な値より緩いM」がLP緩和をどれだけ緩めるかを4節で測る。

## 2. 診断 (`mk.analyze`)

実際に解いて双対境界・空間分枝・数値スケール・対称性を観測し、診断ルールに照らす。

In [3]:
report = mk.analyze(lambda: tep.build_model("default"), name="transmission-default", time_limit=30)
print(report.summary())
print()
print({k: report.metrics.get(k) for k in [
    "gap", "nodes", "nsols", "spatial_share", "has_nonlinear",
    "residual_coef_ratio", "coef_ratio", "residual_bigm_count",
    "n_sym_groups", "largest_sym_group", "sym_sound"]})

[transmission-default] 検出症状 2件:
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
  [good] 入替可能な変数群(対称性)を検出(参考情報) -> 通常は対応不要(SCIPが自動処理)。usesymmetryを無効化した運用でのみ辞書式除去が有効

{'gap': 0.008977608681100768, 'nodes': 1, 'nsols': 2, 'spatial_share': 0.0, 'has_nonlinear': False, 'residual_coef_ratio': 1.5852670688344146e+19, 'coef_ratio': 21120.0, 'residual_bigm_count': 0, 'n_sym_groups': 9, 'largest_sym_group': 5, 'sym_sound': True}


`has_nonlinear=False` — この定式化は真の非凸を含まない**純粋MILP**である(位相角差×
感受率は定数×変数のアフィン式であり非線形項ではない)。それでもSCIPはルート1ノードで
gapを1%未満まで詰める。診断で残るのは **数値スケール**(`numerical_scale`)と
**対称な変数群の検出**(`symmetry_info`、参考情報)の2つ。3節でそれぞれの発火根拠を裏付ける。

## 3. 診断の中身を見る

### 3-1. 数値スケール — `residual_coef_ratio` は何が押し上げているか

`residual_coef_ratio`(presolve後の係数比)を素朴に見ると 1e19 という桁違いの値になるが、
これは「本当に緩いBig-M」のせいなのか、それとも別の要因なのかを実際の係数分布で確認する。

In [4]:
m3 = tep.build_model("default")
pre = residual_scale(m3)
print(f"presolve後: min={pre['min']:.3g} max={pre['max']:.3g} ratio={pre['ratio']:.3g} "
      f"median={pre['median']:.3g}")

m4 = tep.build_model("default")
m4.hideOutput()
m4.presolve()  # residual_scale と同じ: presolve後の係数分布を見る
df_coef = extract_coefficients(m4)
df_sorted = df_coef.sort_values("magnitude")
print("--- 最小側(何がminを作っているか) ---")
print(df_sorted.head(5)[["source", "magnitude", "name"]].to_string(index=False))
print("--- 最大側(何がmaxを作っているか) ---")
print(df_sorted.tail(5)[["source", "magnitude", "name"]].to_string(index=False))

presolve後: min=6.66e-16 max=1.06e+04 ratio=1.59e+19 median=3.35
--- 最小側(何がminを作っているか) ---
source    magnitude       name
  変数境界 6.661338e-16 flow_e_6_2
  変数境界 6.661338e-16 flow_e_6_1
  変数境界 6.661338e-16 flow_e_6_0
  変数境界 6.661338e-16 flow_e_6_3
  変数境界 6.661338e-16 flow_e_6_4
--- 最大側(何がmaxを作っているか) ---
source  magnitude     name
  目的係数    10560.0 shed_0_2
  目的係数    10560.0 shed_1_2
  目的係数    10560.0 shed_2_2
  目的係数    10560.0 shed_3_2
  目的係数    10560.0 shed_4_2


In [5]:
fig = go.Figure(layout=base_layout(
    "presolve後の係数分布(出所別) — 最大は目的係数(計画外停電VOLL)、最小はpresolveの浮動小数点残差",
    "出所", "|係数| (log scale)", height=400))
for src, color in [("制約係数", C["s1"]), ("RHS/LHS", C["muted"]), ("目的係数", C["warn"]),
                   ("変数境界", C["s2"])]:
    sub = df_coef[df_coef["source"] == src]
    if sub.empty:
        continue
    fig.add_trace(go.Box(y=sub["magnitude"], name=src, marker_color=color, boxpoints="outliers"))
fig.update_yaxes(type="log")
show(fig)

最小値(≈6.7e-16)は presolve が制約を集約したときに生じた**浮動小数点の残差**
(理論上ゼロになるはずの項が丸め誤差でわずかに残った値)であり、モデルが本当に持つ
「意味のある最小係数」ではない。意味のある比較は「目的係数の最大(計画外停電コスト
VOLL=8000×シナリオ重みで最大約1万)」と「制約係数(感受率3〜8、容量15〜55)」の間であり、
その比は概ね1e3のオーダーに過ぎない。**`residual_coef_ratio>=1e6` というルールの閾値は
機械的には発火するが、実際の悪条件の主因はpresolveの数値残差であって、対処すべきBig-Mの
緩さではない** — この区別を次にBig-M自体の緩さで確認する。

### 3-2. 対称性 — `symmetry_info` の発火根拠

`detect_symmetry` は「入替可能な変数群」を1-hop シグネチャで検出する。

In [6]:
groups_df, sym_summary = detect_symmetry(tep.build_model("default"))
print(sym_summary)
print(groups_df.sort_values("size", ascending=False).head(6).to_string(index=False))

{'n_linear_vars': 193, 'n_groups': 9, 'n_symmetric_vars': 45, 'largest_group': 5, 'n_nonlinear_conss': 0, 'sound': True, 'has_symmetry': True, 'caveat': None}
 signature_id  size                                               members
            0     5 theta_0_0, theta_0_1, theta_0_2, theta_0_3, theta_0_4
            1     5 theta_1_0, theta_1_1, theta_1_2, theta_1_3, theta_1_4
            2     5 theta_2_0, theta_2_1, theta_2_2, theta_2_3, theta_2_4
            3     5 theta_3_0, theta_3_1, theta_3_2, theta_3_3, theta_3_4
            4     5 theta_4_0, theta_4_1, theta_4_2, theta_4_3, theta_4_4
            5     5 theta_5_0, theta_5_1, theta_5_2, theta_5_3, theta_5_4


In [7]:
top = groups_df.sort_values("size", ascending=False).head(9)
fig = go.Figure(layout=base_layout(
    "対称(入替可能)な変数群 — 最大群のサイズ", "signature_id(サイズ降順)", "群サイズ", height=360))
fig.add_trace(go.Bar(x=[f"群{i}" for i in top["signature_id"]], y=top["size"],
    marker_color=C["s1"], text=top["size"], textposition="outside", cliponaxis=False))
show(fig)

最大群(サイズ5)の中身は、同一バスの位相角 `theta_b_*` が5シナリオ分並んだもの。
これは偶然ではない — `theta` はKirchhoff等式(潮流=感受率×位相角差、ネットワーク
トポロジのみに依存)にしか現れず、需要 `demand[s,b]` に依存する需給バランス式には
直接現れない(そこに現れるのは `flow`/`gen`/`shed`)。したがって同じバスの `theta` は
シナリオが変わっても全く同じ形の制約集合に全く同じ係数で参加し、真に入替可能になる。
一方 `flow`/`shed` はシナリオごとに異なる `demand` を右辺に持つため対称にならない —
「対称性が生まれる/生まれない」の境目が、変数がどの制約(需要依存か否か)に触れるかで
決まることが分かる。ただし severity=good扱いの通り、SCIPは`usesymmetry`既定ONで自動的に
対称性を処理するため、手動の辞書式順序制約は通常不要 —— 実際に nodes=1(ルートのみ)で
解けていることがそれを裏付ける。

## 4. 改善を試す — Big-M排除(`BIG_M=400`は本当に必要な値より何倍緩いか)

`kirchhoff_cand_*` 制約は共通の `BIG_M=400` を使うが、候補線 `c` が実際に取りうる
潮流の絶対値は `b_cand[c] * (θ範囲の全幅)` で抑えられる($\theta$ 範囲は $\pm0.5$ rad なので全幅1.0)。
候補線ごとにタイトな `M_c = b_cand[c] * 1.0` へ置き換えた版を作り、(a) 既定設定
(presolve/cutあり)での実測、(b) 定式化そのものの質(presolve/cutを切った純粋LP緩和)
の両方を正直に比較する。

In [8]:
def build_tight(scale="default"):
    # 候補線ごとのタイトなBig-M(= b_cand[c] * theta全幅)に置き換えた版
    d = tep._data(scale)
    nB, nE, nC, nS = d["nB"], d["nE"], d["nC"], d["nS"]
    existing, cand_edges = d["existing"], d["cand_edges"]
    b_exist, cap_exist = d["b_exist"], d["cap_exist"]
    b_cand, cap_cand, invest_cost = d["b_cand"], d["cap_cand"], d["invest_cost"]
    gen_bus, gen_cap, gen_cost = d["gen_bus"], d["gen_cap"], d["gen_cost"]
    demand, scenario_weight = d["demand"], d["scenario_weight"]
    THETA_MAX = 0.5
    tight_M = b_cand * (2 * THETA_MAX)  # 候補線ごとのタイトなBig-M

    m = Model("Transmission_Expansion_Operation_TightM")
    B, E, Cs, S = range(nB), range(nE), range(nC), range(nS)
    G = range(len(gen_bus))
    REF = 0
    build = {c: m.addVar(vtype="B", name=f"build_{c}") for c in Cs}
    theta, flow_e, flow_c, gen, shed = {}, {}, {}, {}, {}
    for s in S:
        for b_ in B:
            lb = 0.0 if b_ == REF else -THETA_MAX
            ub = 0.0 if b_ == REF else THETA_MAX
            theta[b_, s] = m.addVar(lb=lb, ub=ub, name=f"theta_{b_}_{s}")
        for e in E:
            flow_e[e, s] = m.addVar(lb=-float(cap_exist[e]), ub=float(cap_exist[e]), name=f"flow_e_{e}_{s}")
        for c in Cs:
            flow_c[c, s] = m.addVar(lb=-float(cap_cand[c]), ub=float(cap_cand[c]), name=f"flow_c_{c}_{s}")
        for g in G:
            gen[g, s] = m.addVar(lb=0.0, ub=float(gen_cap[g]), name=f"gen_{g}_{s}")
        for b_ in B:
            shed[b_, s] = m.addVar(lb=0.0, ub=float(demand[s, b_]), name=f"shed_{b_}_{s}")
    for s in S:
        for e, (i, j) in enumerate(existing):
            m.addCons(flow_e[e, s] == float(b_exist[e]) * (theta[i, s] - theta[j, s]))
        for c, (i, j) in enumerate(cand_edges):
            m.addCons(flow_c[c, s] <= float(cap_cand[c]) * build[c])
            m.addCons(flow_c[c, s] >= -float(cap_cand[c]) * build[c])
            Mc = float(tight_M[c])
            m.addCons(flow_c[c, s] - float(b_cand[c]) * (theta[i, s] - theta[j, s]) <= Mc * (1 - build[c]))
            m.addCons(flow_c[c, s] - float(b_cand[c]) * (theta[i, s] - theta[j, s]) >= -Mc * (1 - build[c]))
        for b_ in B:
            inflow = quicksum(flow_e[e, s] for e, (i, j) in enumerate(existing) if j == b_) \
                - quicksum(flow_e[e, s] for e, (i, j) in enumerate(existing) if i == b_) \
                + quicksum(flow_c[c, s] for c, (i, j) in enumerate(cand_edges) if j == b_) \
                - quicksum(flow_c[c, s] for c, (i, j) in enumerate(cand_edges) if i == b_)
            local_gen = quicksum(gen[g, s] for g in G if int(gen_bus[g]) == b_)
            m.addCons(local_gen + inflow + shed[b_, s] == float(demand[s, b_]))
    invest = quicksum(float(invest_cost[c]) * build[c] for c in Cs)
    operation = quicksum(float(scenario_weight[s]) * (
        quicksum(float(gen_cost[g]) * gen[g, s] for g in G)
        + quicksum(tep.VOLL * shed[b_, s] for b_ in B)) for s in S)
    m.setObjective(invest + operation, "minimize")
    m.data = dict(build=build)
    return m

print("loose M(共通400)   :", tep.BIG_M)
tight_vals = tep._data("default")["b_cand"] * 1.0
print("tight M(候補線ごと) :", np.round(tight_vals, 2), "-- 最大でも", round(float(tight_vals.max()), 2))

loose M(共通400)   : 400.0
tight M(候補線ごと) : [7.08 6.52 6.03 4.72 6.82 7.41 5.77 7.5 ] -- 最大でも 7.5


In [9]:
df = mk.compare_variants(
    {"loose-M(400共通)": lambda: tep.build_model("default"),
     "tight-M(候補線ごと)": build_tight}, time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,loose-M(400共通),3.704492e+06,3.701099e+06,0.008978,1
1,tight-M(候補線ごと),3.704492e+06,3.697824e+06,0.009871,1


既定設定(presolve/cutあり)では両者の `root_dual` / `final_gap` / `nodes` はほぼ同一に
なる。SCIPのpresolveが自明な範囲でBig-Mを自動的に締めるため、既定運用では
「400→候補線ごとの最大8.0」という17〜50倍のタイト化がほとんど効果を持たない
(2節で見た nodes=1 の速さも、presolveがすでに実質的にタイトな緩和を作れているため)。
定式化そのものの質の差は、presolve/cutを切った純粋LP緩和でのみ現れる。

In [10]:
def root_lp_bound(build_fn):
    m = build_fn()
    m.hideOutput()
    m.setPresolve(SCIP_PARAMSETTING.OFF)
    m.setSeparating(SCIP_PARAMSETTING.OFF)
    m.setHeuristics(SCIP_PARAMSETTING.OFF)
    m.setParam("limits/nodes", 1)
    m.optimize()
    return m.getDualbound()

loose_lp = root_lp_bound(lambda: tep.build_model("default"))
tight_lp = root_lp_bound(build_tight)
print(f"純粋LP緩和(presolve/cut/heuristics off): loose-M={loose_lp:.1f}  tight-M={tight_lp:.1f}  "
      f"差={(tight_lp - loose_lp):+.1f}({(tight_lp - loose_lp) / abs(loose_lp) * 100:+.1f}%)")

純粋LP緩和(presolve/cut/heuristics off): loose-M=3493082.1  tight-M=3485575.1  差=-7507.1(-0.2%)


In [11]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15,
    subplot_titles=("既定設定のroot_dual(presolve/cutあり) — ほぼ差なし",
                    "純粋LP緩和(presolve/cut/heuristics off) — 定式化の質の差"))
loose_row = df.loc[df["variant"] == "loose-M(400共通)"].iloc[0]
tight_row = df.loc[df["variant"] == "tight-M(候補線ごと)"].iloc[0]
fig.add_trace(go.Bar(x=["loose-M", "tight-M"], y=[loose_row["root_dual"], tight_row["root_dual"]],
    marker_color=[C["muted"], C["s1"]], text=[f"{v:.0f}" for v in
    [loose_row["root_dual"], tight_row["root_dual"]]], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=["loose-M", "tight-M"], y=[loose_lp, tight_lp],
    marker_color=[C["muted"], C["s1"]], text=[f"{v:.0f}" for v in [loose_lp, tight_lp]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=2)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="Big-M排除の効果 — 既定設定 vs 純粋LP緩和", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

## まとめ

- この問題の本質的な難しさは「候補線増強(整数)が既存より安価な発電地点への物理的
  経路を作るかどうか」を、5通りの需要シナリオ全てで同時に満たす投資選択にある
  (2段階ロバスト計画)。真の非凸を含まない純粋MILPだが、disjunctive(big-M)構造ゆえに
  整数決定が連続変数の可行領域自体を変える。
- SCIPのpresolve/内蔵対称性処理が強力なため、この規模ではgap・ノード数自体は診断の
  題材にならない(root 1ノードで gap<1%)。**診断で拾うべきは「presolveが締めてくれる分」
  と「本質的に必要な分」の区別**であり、`residual_coef_ratio` のような機械的な閾値判定は
  presolveの数値残差(浮動小数点誤差)を誤って"悪条件"と report しうることを3節で示した。
- Big-M自体は`400`→候補線ごとの最大`8.0`という50倍のタイト化余地があったが、
  既定設定での実測にはほぼ効果がなかった(presolveが実質的に同じ仕事をする)。
  効果は presolve/cut/heuristics を切った純粋LP緩和でのみ観測できる —
  「Big-Mが効かないとき」の典型例。

### この診断が対応する実務の意思決定

系統計画者が投資委員会に提出する「どの候補コリドーを増強すべきか」という提案は、
単一シナリオでは容易でも、複数シナリオ間のトレードオフが生じた瞬間にNP困難になる。
本notebookの診断は「この規模ならSCIPの既定設定で十分速く解けるので、複雑な前処理
(Big-M手動排除)への投資対効果は低い」という、計画作業そのものへの実務的な判断材料になる。

関連: [手法ガイド 3. Big-M/Indicator](../../playbook/03-bigm.md) /
[08. 条件数・数値健全性](../../playbook/08-condition.md) /
サンプルカタログ [立地・ネットワーク設計](../../samples/location_and_network_design.md)